In [8]:
# import pandas as pd
# import numpy as np
# import tensorflow as tf
# import os
# import logging
# import re
# import ast
# import pickle
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
# from tensorflow.keras.preprocessing.text import Tokenizer
# from tensorflow.keras.preprocessing.sequence import pad_sequences
# from tensorflow.keras.layers import Input, Dense, Embedding, Flatten, Concatenate, Dropout
# from tensorflow.keras.models import Model
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.regularizers import l2
# from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# # Set up logging
# logging.basicConfig(
#     level=logging.INFO,
#     format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
#     handlers=[
#         logging.FileHandler('beauty_recommender.log'),
#         logging.StreamHandler()
#     ]
# )
# logger = logging.getLogger('beauty_recommender')

# # Set TensorFlow environment
# os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

# class BeautyRecommender:
#     """Beauty product recommendation system using a dual tower architecture"""
    
#     def __init__(self, embedding_dim=128, tower_dim=64, max_user_words=10000, 
#                  max_product_words=10000, max_user_len=200, max_product_len=200):
#         """Initialize the recommender with configuration parameters"""
#         self.embedding_dim = embedding_dim
#         self.tower_dim = tower_dim
#         self.max_user_words = max_user_words
#         self.max_product_words = max_product_words
#         self.max_user_len = max_user_len
#         self.max_product_len = max_product_len
        
#         # Initialize containers for model components
#         self.user_tokenizer = None
#         self.product_tokenizer = None
#         self.category_encoders = {}
#         self.numerical_scaler = None
#         self.user_numerical_scaler = None
#         self.product_numerical_scaler = None
#         self.model = None
#         self.user_embedding_model = None
#         self.product_embedding_model = None
        
#         # Track feature dimensions
#         self.feature_dimensions = {}
        
#     def load_data(self, product_path, review_path):
#         """Load product and review data from CSV files"""
#         logger.info(f"Loading data from {product_path} and {review_path}")
#         try:
#             products_df = pd.read_csv(product_path, encoding='ISO-8859-1')
#             reviews_df = pd.read_csv(review_path)
            
#             # Log data dimensions
#             logger.info(f"Product data shape: {products_df.shape}")
#             logger.info(f"Review data shape: {reviews_df.shape}")
            
#             # Check for critical columns
#             required_product_cols = ['product_id', 'price_usd', 'sale_price_usd']
#             required_review_cols = ['product_id', 'review_text', 'review_title', 'rating']
            
#             for col in required_product_cols:
#                 if col not in products_df.columns:
#                     logger.warning(f"Missing critical column in products data: {col}")
            
#             for col in required_review_cols:
#                 if col not in reviews_df.columns:
#                     logger.warning(f"Missing critical column in reviews data: {col}")
            
#             # Calculate discount percentage
#             products_df['discount_pct'] = products_df.apply(
#                 lambda row: ((row['price_usd'] - row['sale_price_usd']) / row['price_usd'] * 100) 
#                 if pd.notna(row['price_usd']) and pd.notna(row['sale_price_usd']) and row['price_usd'] > 0 
#                 else 0, 
#                 axis=1
#             )
#             products_df['discount_pct'] = products_df['discount_pct'].fillna(0).clip(lower=0)
            
#             logger.info(f"Loaded {len(products_df)} products and {len(reviews_df)} reviews")
#             return products_df, reviews_df
#         except Exception as e:
#             logger.error(f"Error loading data: {str(e)}")
#             raise
    
#     def preprocess_data(self, products_df, reviews_df):
#         """Preprocess and merge product and review data"""
#         logger.info("Preprocessing data...")
        
#         # Make copies to avoid modifying original dataframes
#         products_df = products_df.copy()
#         reviews_df = reviews_df.copy()
        
#         # Ensure product_id is of the same type in both dataframes
#         products_df['product_id'] = products_df['product_id'].astype(str)
#         reviews_df['product_id'] = reviews_df['product_id'].astype(str)
        
#         # Log counts before merge
#         product_count = len(products_df)
#         review_count = len(reviews_df)
#         logger.info(f"Products before merge: {product_count}")
#         logger.info(f"Reviews before merge: {review_count}")
        
#         # Merge product and review data
#         merged_df = pd.merge(reviews_df, products_df, on='product_id', how='inner')
#         logger.info(f"Merged dataset size: {len(merged_df)} rows")
        
#         # Check if merge resulted in empty dataframe
#         if len(merged_df) == 0:
#             logger.error("Merge resulted in empty dataframe. Check product_id values in both dataframes.")
#             # Print sample of product_ids from both dataframes to help diagnose
#             logger.error(f"Sample product_ids from products: {products_df['product_id'].head().tolist()}")
#             logger.error(f"Sample product_ids from reviews: {reviews_df['product_id'].head().tolist()}")
#             raise ValueError("Merge resulted in empty dataframe")
        
#         # Clean text columns
#         merged_df['review_text'] = merged_df['review_text'].fillna('')
#         merged_df['review_title'] = merged_df['review_title'].fillna('')
        
#         # Process product highlights
#         merged_df['highlights'] = merged_df['highlights'].fillna('[]')
#         try:
#             merged_df['highlights'] = merged_df['highlights'].apply(
#                 lambda x: ast.literal_eval(x) if isinstance(x, str) else []
#             )
#             merged_df['highlights_text'] = merged_df['highlights'].apply(lambda x: ' '.join(x))
#         except Exception as e:
#             logger.warning(f"Error processing highlights: {str(e)}. Using empty highlights.")
#             merged_df['highlights'] = merged_df['highlights'].apply(lambda x: [])
#             merged_df['highlights_text'] = ''
        
#         # Clean numerical fields
#         numerical_columns = ['loves_count', 'rating_x', 'price_usd_y', 'discount_pct']
#         for col in numerical_columns:
#             if col in merged_df.columns:
#                 merged_df[col] = merged_df[col].fillna(0)
        
#         # Handle categorical data
#         categorical_columns = ['primary_category', 'secondary_category', 'tertiary_category', 
#                               'brand_name_x', 'skin_type', 'skin_tone']
#         for col in categorical_columns:
#             if col in merged_df.columns:
#                 merged_df[col] = merged_df[col].fillna('Unknown')
        
#         # Add text features
#         merged_df['clean_review_text'] = self._basic_text_cleaning(
#             merged_df['review_text'] + ' ' + merged_df['review_title']
#         )
#         merged_df['clean_product_text'] = self._basic_text_cleaning(
#             merged_df['product_name_x'] + ' ' + merged_df['highlights_text']
#         )
        
#         return merged_df
    
#     def _basic_text_cleaning(self, text_series):
#         """Basic text cleaning without external dependencies"""
#         logger.info("Performing basic text cleaning")
        
#         def clean_text(text):
#             if not isinstance(text, str):
#                 return ''
            
#             # Convert to lowercase
#             text = text.lower()
            
#             # Remove special characters and digits
#             text = re.sub(r'[^\w\s]', ' ', text)
#             text = re.sub(r'\d+', ' ', text)
            
#             # Remove extra whitespace
#             text = re.sub(r'\s+', ' ', text)
#             return text.strip()
        
#         return text_series.apply(clean_text)
    
#     def tokenize_text(self, df):
#         """Tokenize user and product text"""
#         logger.info("Tokenizing text...")
        
#         # Validate input columns
#         if 'clean_review_text' not in df.columns or 'clean_product_text' not in df.columns:
#             logger.error("Required text columns not found in dataframe")
#             raise ValueError("Required columns 'clean_review_text' or 'clean_product_text' not found")
        
#         # User text tokenizer
#         self.user_tokenizer = Tokenizer(num_words=self.max_user_words)
#         self.user_tokenizer.fit_on_texts(df['clean_review_text'])
#         user_sequences = self.user_tokenizer.texts_to_sequences(df['clean_review_text'])
#         user_padded = pad_sequences(user_sequences, maxlen=self.max_user_len, padding='post')
        
#         # Product text tokenizer
#         self.product_tokenizer = Tokenizer(num_words=self.max_product_words)
#         self.product_tokenizer.fit_on_texts(df['clean_product_text'])
#         product_sequences = self.product_tokenizer.texts_to_sequences(df['clean_product_text'])
#         product_padded = pad_sequences(product_sequences, maxlen=self.max_product_len, padding='post')
        
#         logger.info(f"User vocabulary size: {len(self.user_tokenizer.word_index)}")
#         logger.info(f"Product vocabulary size: {len(self.product_tokenizer.word_index)}")
#         logger.info(f"User padded shape: {user_padded.shape}")
#         logger.info(f"Product padded shape: {product_padded.shape}")
        
#         return user_padded, product_padded
    
#     def encode_categorical_features(self, df):
#         """Encode categorical features"""
#         logger.info("Encoding categorical features...")
        
#         encoded_features = {}
        
#         # User categorical features (user profile related)
#         user_categorical_columns = ['skin_type', 'skin_tone']
#         user_categorical_features = []
        
#         for col in user_categorical_columns:
#             if col in df.columns:
#                 encoder = LabelEncoder()
#                 encoded_features[col] = encoder.fit_transform(df[col])
#                 self.category_encoders[col] = encoder
#                 user_categorical_features.append(encoded_features[col])
        
#         # Product categorical features
#         product_categorical_columns = ['primary_category', 'secondary_category', 'tertiary_category', 'brand_name_x']
#         product_categorical_features = []
        
#         for col in product_categorical_columns:
#             if col in df.columns:
#                 encoder = LabelEncoder()
#                 encoded_features[col] = encoder.fit_transform(df[col])
#                 self.category_encoders[col] = encoder
#                 product_categorical_features.append(encoded_features[col])
        
#         # Stack features to create input arrays
#         user_categorical_stack = np.column_stack(user_categorical_features) if user_categorical_features else np.zeros((len(df), 1))
#         product_categorical_stack = np.column_stack(product_categorical_features) if product_categorical_features else np.zeros((len(df), 1))
        
#         self.feature_dimensions['user_categorical'] = user_categorical_stack.shape[1]
#         self.feature_dimensions['product_categorical'] = product_categorical_stack.shape[1]
        
#         logger.info(f"User categorical features shape: {user_categorical_stack.shape}")
#         logger.info(f"Product categorical features shape: {product_categorical_stack.shape}")
        
#         return user_categorical_stack, product_categorical_stack
    
#     def scale_numerical_features(self, df):
#         """Scale numerical features and ensure balanced feature sets"""
#         logger.info("Scaling numerical features...")
        
#         # User-related numerical features
#         user_numerical_columns = ['rating_x']
        
#         # Product-related numerical features
#         product_numerical_columns = ['loves_count', 'price_usd_y', 'discount_pct']
        
#         # Ensure all columns exist
#         user_numerical_columns = [col for col in user_numerical_columns if col in df.columns]
#         product_numerical_columns = [col for col in product_numerical_columns if col in df.columns]
        
#         # Log the columns being used
#         logger.info(f"User numerical columns: {user_numerical_columns}")
#         logger.info(f"Product numerical columns: {product_numerical_columns}")
        
#         # If we have no numerical features, create dummy ones
#         if not user_numerical_columns:
#             df['user_num_dummy'] = 0
#             user_numerical_columns = ['user_num_dummy']
        
#         if not product_numerical_columns:
#             df['product_num_dummy'] = 0
#             product_numerical_columns = ['product_num_dummy']
        
#         # Extract and scale user numerical features
#         user_numerical_features = df[user_numerical_columns].fillna(0).values
#         product_numerical_features = df[product_numerical_columns].fillna(0).values
        
#         # Check for NaN values after filling
#         if np.isnan(user_numerical_features).any():
#             logger.warning("NaN values found in user numerical features after filling")
#             user_numerical_features = np.nan_to_num(user_numerical_features)
        
#         if np.isnan(product_numerical_features).any():
#             logger.warning("NaN values found in product numerical features after filling")
#             product_numerical_features = np.nan_to_num(product_numerical_features)
        
#         # Scale features
#         self.user_numerical_scaler = StandardScaler()
#         self.product_numerical_scaler = StandardScaler()
        
#         scaled_user_numerical = self.user_numerical_scaler.fit_transform(user_numerical_features)
#         scaled_product_numerical = self.product_numerical_scaler.fit_transform(product_numerical_features)
        
#         self.feature_dimensions['user_numerical'] = scaled_user_numerical.shape[1]
#         self.feature_dimensions['product_numerical'] = scaled_product_numerical.shape[1]
        
#         logger.info(f"User numerical features shape: {scaled_user_numerical.shape}")
#         logger.info(f"Product numerical features shape: {scaled_product_numerical.shape}")
        
#         return scaled_user_numerical, scaled_product_numerical
    
#     def build_model(self):
#         """Build the dual tower recommendation model"""
#         logger.info("Building dual tower model...")
        
#         # Validate feature dimensions
#         for key in ['user_categorical', 'user_numerical', 'product_categorical', 'product_numerical']:
#             if key not in self.feature_dimensions:
#                 logger.error(f"Missing feature dimension for {key}")
#                 raise ValueError(f"Missing feature dimension for {key}")
        
#         # User Tower - Text Input
#         user_text_input = Input(shape=(self.max_user_len,), name='user_text_input')
#         user_embedding = Embedding(
#             input_dim=min(len(self.user_tokenizer.word_index) + 1, self.max_user_words),
#             output_dim=self.embedding_dim
#         )(user_text_input)
#         user_text_features = Flatten()(user_embedding)
        
#         # User Tower - Categorical Features Input
#         user_categorical_input = Input(shape=(self.feature_dimensions['user_categorical'],), name='user_categorical_input')
#         user_categorical_features = Dense(32, activation='relu')(user_categorical_input)
        
#         # User Tower - Numerical Features Input
#         user_numerical_input = Input(shape=(self.feature_dimensions['user_numerical'],), name='user_numerical_input')
#         user_numerical_features = Dense(32, activation='relu')(user_numerical_input)
        
#         # User Tower - Combined Features
#         user_combined = Concatenate()([
#             user_text_features,
#             user_categorical_features,
#             user_numerical_features
#         ])
        
#         user_dense_1 = Dense(256, activation='relu', kernel_regularizer=l2(0.01))(user_combined)
#         user_dropout = Dropout(0.3)(user_dense_1)
#         user_dense_2 = Dense(self.tower_dim, activation='relu', kernel_regularizer=l2(0.01))(user_dropout)
#         user_tower = tf.math.l2_normalize(user_dense_2, axis=1)
        
#         # Product Tower - Text Input
#         product_text_input = Input(shape=(self.max_product_len,), name='product_text_input')
#         product_embedding = Embedding(
#             input_dim=min(len(self.product_tokenizer.word_index) + 1, self.max_product_words),
#             output_dim=self.embedding_dim
#         )(product_text_input)
#         product_text_features = Flatten()(product_embedding)
        
#         # Product Tower - Categorical Features Input
#         product_categorical_input = Input(shape=(self.feature_dimensions['product_categorical'],), name='product_categorical_input')
#         product_categorical_features = Dense(32, activation='relu')(product_categorical_input)
        
#         # Product Tower - Numerical Features Input
#         product_numerical_input = Input(shape=(self.feature_dimensions['product_numerical'],), name='product_numerical_input')
#         product_numerical_features = Dense(32, activation='relu')(product_numerical_input)
        
#         # Product Tower - Combined Features
#         product_combined = Concatenate()([
#             product_text_features,
#             product_categorical_features,
#             product_numerical_features
#         ])
        
#         product_dense_1 = Dense(256, activation='relu', kernel_regularizer=l2(0.01))(product_combined)
#         product_dropout = Dropout(0.3)(product_dense_1)
#         product_dense_2 = Dense(self.tower_dim, activation='relu', kernel_regularizer=l2(0.01))(product_dropout)
#         product_tower = tf.math.l2_normalize(product_dense_2, axis=1)
        
#         # Dot product for similarity
#         dot_product = tf.reduce_sum(tf.multiply(user_tower, product_tower), axis=1, keepdims=True)
        
#         # Build model
#         self.model = Model(
#             inputs=[
#                 user_text_input,
#                 user_categorical_input,
#                 user_numerical_input,
#                 product_text_input,
#                 product_categorical_input,
#                 product_numerical_input
#             ],
#             outputs=dot_product
#         )
        
#         # Compile model
#         self.model.compile(
#             optimizer=Adam(learning_rate=0.001),
#             loss='binary_crossentropy',
#             metrics=['accuracy']
#         )
        
#         logger.info(f"Model built with {self.model.count_params()} parameters")
        
#         # Explicitly create embedding models
#         self.user_embedding_model = Model(
#             inputs=[
#                 user_text_input,
#                 user_categorical_input,
#                 user_numerical_input
#             ],
#             outputs=user_tower
#         )
        
#         self.product_embedding_model = Model(
#             inputs=[
#                 product_text_input,
#                 product_categorical_input,
#                 product_numerical_input
#             ],
#             outputs=product_tower
#         )
        
#         return self.model
    
#     def generate_training_data(self, user_padded, user_categorical, user_numerical,
#                               product_padded, product_categorical, product_numerical,
#                               negative_ratio=4, max_samples=None):
#         """Generate training data with positive and negative examples"""
#         logger.info(f"Generating training data with negative_ratio={negative_ratio}")
        
#         # Validate input shapes
#         logger.info(f"User padded shape: {user_padded.shape}")
#         logger.info(f"User categorical shape: {user_categorical.shape}")
#         logger.info(f"User numerical shape: {user_numerical.shape}")
#         logger.info(f"Product padded shape: {product_padded.shape}")
#         logger.info(f"Product categorical shape: {product_categorical.shape}")
#         logger.info(f"Product numerical shape: {product_numerical.shape}")
        
#         # Ensure all input arrays have the same first dimension
#         if not (len(user_padded) == len(user_categorical) == len(user_numerical) == 
#                 len(product_padded) == len(product_categorical) == len(product_numerical)):
#             logger.error("Input arrays have inconsistent lengths")
#             raise ValueError("All input arrays must have the same number of samples")
        
#         n_samples = len(user_padded)
#         if max_samples and max_samples < n_samples:
#             n_samples = max_samples
#             logger.info(f"Limiting to {max_samples} samples")
        
#         # Estimate total size after adding negative examples
#         total_samples = n_samples * (1 + negative_ratio)
#         logger.info(f"Estimated total samples after negative sampling: {total_samples}")
        
#         # Allocate arrays for efficiency
#         X_user_text = np.zeros((total_samples, user_padded.shape[1]), dtype=np.int32)
#         X_user_cat = np.zeros((total_samples, user_categorical.shape[1]), dtype=np.float32)
#         X_user_num = np.zeros((total_samples, user_numerical.shape[1]), dtype=np.float32)
#         X_product_text = np.zeros((total_samples, product_padded.shape[1]), dtype=np.int32)
#         X_product_cat = np.zeros((total_samples, product_categorical.shape[1]), dtype=np.float32)
#         X_product_num = np.zeros((total_samples, product_numerical.shape[1]), dtype=np.float32)
#         y = np.zeros((total_samples, 1), dtype=np.float32)
        
#         # Track sample index
#         sample_idx = 0
        
#         # Add positive samples
#         for i in range(n_samples):
#             # Positive example
#             X_user_text[sample_idx] = user_padded[i]
#             X_user_cat[sample_idx] = user_categorical[i]
#             X_user_num[sample_idx] = user_numerical[i]
#             X_product_text[sample_idx] = product_padded[i]
#             X_product_cat[sample_idx] = product_categorical[i]
#             X_product_num[sample_idx] = product_numerical[i]
#             y[sample_idx] = 1  # Positive label
#             sample_idx += 1
            
#             # Generate negative examples
#             for _ in range(negative_ratio):
#                 # Select a random product that is different from the current one
#                 neg_idx = np.random.randint(0, n_samples)
#                 while neg_idx == i and n_samples > 1:  # Ensure different product if possible
#                     neg_idx = np.random.randint(0, n_samples)
                
#                 # Add negative example
#                 X_user_text[sample_idx] = user_padded[i]  # Same user
#                 X_user_cat[sample_idx] = user_categorical[i]  # Same user attributes
#                 X_user_num[sample_idx] = user_numerical[i]  # Same user numerical features
#                 X_product_text[sample_idx] = product_padded[neg_idx]  # Different product
#                 X_product_cat[sample_idx] = product_categorical[neg_idx]  # Different product attributes
#                 X_product_num[sample_idx] = product_numerical[neg_idx]  # Different product numerical features
#                 y[sample_idx] = 0  # Negative label
#                 sample_idx += 1
                
#             # Log progress every 1000 samples
#             if i > 0 and i % 1000 == 0:
#                 logger.info(f"Generated {sample_idx} training samples from {i}/{n_samples} original samples")
        
#         # Verify final sample count
#         actual_samples = sample_idx
#         logger.info(f"Generated {actual_samples} training samples")
        
#         # Trim arrays if we didn't use all allocated space
#         if actual_samples < total_samples:
#             X_user_text = X_user_text[:actual_samples]
#             X_user_cat = X_user_cat[:actual_samples]
#             X_user_num = X_user_num[:actual_samples]
#             X_product_text = X_product_text[:actual_samples]
#             X_product_cat = X_product_cat[:actual_samples]
#             X_product_num = X_product_num[:actual_samples]
#             y = y[:actual_samples]
        
#         # Create training data arrays
#         train_data = [
#             X_user_text,
#             X_user_cat,
#             X_user_num,
#             X_product_text,
#             X_product_cat,
#             X_product_num
#         ]
        
#         return train_data, y
    
#     def train_model(self, train_data, train_labels, validation_split=0.2, epochs=10, batch_size=32):
#         """Train the recommendation model"""
#         logger.info(f"Training model with {len(train_labels)} samples, validation_split={validation_split}")
        
#         # Validate model
#         if self.model is None:
#             logger.error("Model not built. Call build_model() first.")
#             raise ValueError("Model not built")
        
#         # Split indices into train and validation
#         indices = np.arange(len(train_labels))
#         train_indices, val_indices = train_test_split(
#             indices,
#             test_size=validation_split,
#             random_state=42,
#             stratify=train_labels if len(np.unique(train_labels)) > 1 else None
#         )
        
#         # Use indices to split data
#         X_train = [x[train_indices] for x in train_data]
#         X_val = [x[val_indices] for x in train_data]
#         y_train = train_labels[train_indices]
#         y_val = train_labels[val_indices]
        
#         # Log training and validation sizes
#         logger.info(f"Training samples: {len(y_train)}")
#         logger.info(f"Validation samples: {len(y_val)}")
        
#         # Define callbacks
#         callbacks = [
#             EarlyStopping(
#                 monitor='val_loss',
#                 patience=3,
#                 restore_best_weights=True
#             ),
#             ReduceLROnPlateau(
#                 monitor='val_loss',
#                 factor=0.5,
#                 patience=2
#             )
#         ]
        
#         # Train the model
#         try:
#             history = self.model.fit(
#                 X_train,
#                 y_train,
#                 validation_data=(X_val, y_val),
#                 epochs=epochs,
#                 batch_size=batch_size,
#                 callbacks=callbacks
#             )
            
#             logger.info("Model training completed")
#             return history
#         except Exception as e:
#             logger.error(f"Error during model training: {str(e)}")
#             raise
    
#     def generate_embeddings(self, user_padded, user_categorical, user_numerical,
#                             product_padded, product_categorical, product_numerical,
#                             batch_size=64):
#         """Generate embeddings for users and products"""
#         logger.info("Generating embeddings...")
        
#         # Validate embedding models
#         if self.user_embedding_model is None or self.product_embedding_model is None:
#             logger.error("Embedding models not available. Call build_model() or load_models() first.")
#             raise ValueError("Embedding models not available")
        
#         # Check input dimensions
#         if (user_padded.shape[0] != user_categorical.shape[0] or 
#             user_padded.shape[0] != user_numerical.shape[0]):
#             logger.error("User input arrays have inconsistent lengths")
#             raise ValueError("User input arrays must have the same number of samples")
        
#         if (product_padded.shape[0] != product_categorical.shape[0] or 
#             product_padded.shape[0] != product_numerical.shape[0]):
#             logger.error("Product input arrays have inconsistent lengths")
#             raise ValueError("Product input arrays must have the same number of samples")
        
#         # Generate user embeddings
#         try:
#             user_embeddings = self.user_embedding_model.predict(
#                 [user_padded, user_categorical, user_numerical],
#                 batch_size=batch_size
#             )
            
#             # Generate product embeddings
#             product_embeddings = self.product_embedding_model.predict(
#                 [product_padded, product_categorical, product_numerical],
#                 batch_size=batch_size
#             )
            
#             # Check that the outputs have expected shapes
#             logger.info(f"Generated {len(user_embeddings)} user embeddings and {len(product_embeddings)} product embeddings")
#             logger.info(f"User embedding dimension: {user_embeddings.shape}")
#             logger.info(f"Product embedding dimension: {product_embeddings.shape}")
            
#             return user_embeddings, product_embeddings
#         except Exception as e:
#             logger.error(f"Error generating embeddings: {str(e)}")
#             raise
    
#     def get_recommendations(self, user_embedding, product_embeddings, df, top_n=5):
#         """Get recommendations for a user based on their embedding"""
#         logger.info("Getting recommendations...")
        
#         # Validate inputs
#         if user_embedding.shape[1] != product_embeddings.shape[1]:
#             logger.error(f"Embedding dimensions mismatch: user {user_embedding.shape}, product {product_embeddings.shape}")
#             raise ValueError("User and product embedding dimensions must match")
        
#         # Calculate similarity scores
#         similarity_scores = np.dot(user_embedding, product_embeddings.T)
        
#         # Make sure we only consider indices that are within the DataFrame's range
#         valid_indices = np.arange(min(len(similarity_scores[0]), len(df)))
        
#         # Get scores only for valid indices
#         valid_scores = similarity_scores[0][valid_indices]
        
#         # Sort the valid indices by their scores
#         sorted_idx = np.argsort(valid_scores)[::-1]
        
#         # Take the top N
#         top_indices = valid_indices[sorted_idx[:top_n]]
        
#         logger.info(f"Selected top {len(top_indices)} products from {len(valid_indices)} candidates")
        
#         # Get unique products (assuming df is already deduplicated by product)
#         try:
#             recommended_products = df.iloc[top_indices]
            
#             # Get relevant product columns for recommendations
#             product_cols = ['product_id', 'product_name_x', 'brand_name_x', 'price_usd_y', 'rating_x']
#             existing_cols = [col for col in product_cols if col in recommended_products.columns]
            
#             logger.info(f"Returning recommendations with columns: {existing_cols}")
            
#             return recommended_products[existing_cols]
#         except Exception as e:
#             logger.error(f"Error getting recommendations: {str(e)}")
#             raise
    
#     def save_models(self, base_path='.'):
#         """Save all models and artifacts to disk"""
#         logger.info(f"Saving models and artifacts to {base_path}")
        
#         # Create directory if it doesn't exist
#         os.makedirs(base_path, exist_ok=True)
        
#         # Save main model
#         if self.model:
#             self.model.save(os.path.join(base_path, 'dual_tower_model.h5'))
        
#         # Save embedding models
#         if self.user_embedding_model:
#             self.user_embedding_model.save(os.path.join(base_path, 'user_embedding_model.h5'))
        
#         if self.product_embedding_model:
#             self.product_embedding_model.save(os.path.join(base_path, 'product_embedding_model.h5'))
        
#         # Save tokenizers
#         if self.user_tokenizer:
#             with open(os.path.join(base_path, 'user_tokenizer.pickle'), 'wb') as handle:
#                 pickle.dump(self.user_tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
        
#         if self.product_tokenizer:
#             with open(os.path.join(base_path, 'product_tokenizer.pickle'), 'wb') as handle:
#                 pickle.dump(self.product_tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
        
#         # Save encoders and scalers
#         if self.category_encoders:
#             with open(os.path.join(base_path, 'category_encoders.pickle'), 'wb') as handle:
#                 pickle.dump(self.category_encoders, handle, protocol=pickle.HIGHEST_PROTOCOL)
        
#         if self.user_numerical_scaler:
#             with open(os.path.join(base_path, 'user_numerical_scaler.pickle'), 'wb') as handle:
#                 pickle.dump(self.user_numerical_scaler, handle, protocol=pickle.HIGHEST_PROTOCOL)
        
#         if self.product_numerical_scaler:
#             with open(os.path.join(base_path, 'product_numerical_scaler.pickle'), 'wb') as handle:
#                 pickle.dump(self.product_numerical_scaler, handle, protocol=pickle.HIGHEST_PROTOCOL)
        
#         # Save feature dimensions
#         with open(os.path.join(base_path, 'feature_dimensions.pickle'), 'wb') as handle:
#             pickle.dump(self.feature_dimensions, handle, protocol=pickle.HIGHEST_PROTOCOL)
        
#         logger.info("Successfully saved all models and artifacts")
    
#     def load_models(self, base_path='.'):
#         """Load all models and artifacts from disk"""
#         logger.info(f"Loading models and artifacts from {base_path}")
        
#         try:
#             # Load feature dimensions
#             with open(os.path.join(base_path, 'feature_dimensions.pickle'), 'rb') as handle:
#                 self.feature_dimensions = pickle.load(handle)
            
#             # Load tokenizers
#             with open(os.path.join(base_path, 'user_tokenizer.pickle'), 'rb') as handle:
#                 self.user_tokenizer = pickle.load(handle)
            
#             with open(os.path.join(base_path, 'product_tokenizer.pickle'), 'rb') as handle:
#                 self.product_tokenizer = pickle.load(handle)
            
#             # Load encoders and scalers
#             with open(os.path.join(base_path, 'category_encoders.pickle'), 'rb') as handle:
#                 self.category_encoders = pickle.load(handle)
            
#             with open(os.path.join(base_path, 'user_numerical_scaler.pickle'), 'rb') as handle:
#                 self.user_numerical_scaler = pickle.load(handle)
            
#             with open(os.path.join(base_path, 'product_numerical_scaler.pickle'), 'rb') as handle:
#                 self.product_numerical_scaler = pickle.load(handle)
            
#             # Load models
#             self.model = tf.keras.models.load_model(os.path.join(base_path, 'dual_tower_model.h5'))
#             self.user_embedding_model = tf.keras.models.load_model(os.path.join(base_path, 'user_embedding_model.h5'))
#             self.product_embedding_model = tf.keras.models.load_model(os.path.join(base_path, 'product_embedding_model.h5'))
            
#             logger.info("Successfully loaded all models and artifacts")
#         except Exception as e:
#             logger.error(f"Error loading models: {str(e)}")
#             raise


# def main():
#     """Main function to run the recommendation system"""
#     try:
#         # Initialize the recommender
#         recommender = BeautyRecommender()
        
#         # Load data
#         products_df, reviews_df = recommender.load_data(
#             '../data/product_info.csv',
#             '../data/reviews_0-250_chunk_0.csv'
#         )
        
#         # Preprocess data
#         processed_df = recommender.preprocess_data(products_df, reviews_df)
        
#         # Feature engineering
#         user_padded, product_padded = recommender.tokenize_text(processed_df)
#         user_categorical, product_categorical = recommender.encode_categorical_features(processed_df)
#         user_numerical, product_numerical = recommender.scale_numerical_features(processed_df)
        
#         # Build model
#         model = recommender.build_model()
        
#         # Generate training data
#         train_data, train_labels = recommender.generate_training_data(
#             user_padded, user_categorical, user_numerical,
#             product_padded, product_categorical, product_numerical,
#             negative_ratio=4,
#             max_samples=10000  # Limit sample size for faster training
#         )
        
#         # Train model
#         history = recommender.train_model(
#             train_data,
#             train_labels,
#             validation_split=0.2,
#             epochs=10,
#             batch_size=32
#         )
        
#         # Create a deduplicated dataframe for recommendations
#         unique_products_df = processed_df.drop_duplicates(subset=['product_id']).reset_index(drop=True)
        
#         # Extract features for the unique products
#         unique_product_padded = recommender.product_tokenizer.texts_to_sequences(unique_products_df['clean_product_text'])
#         unique_product_padded = pad_sequences(unique_product_padded, maxlen=recommender.max_product_len, padding='post')
        
#         # Re-encode categorical features for unique products
#         unique_product_categorical = []
#         product_categorical_columns = ['primary_category', 'secondary_category', 'tertiary_category', 'brand_name_x']
#         for col in product_categorical_columns:
#             if col in unique_products_df.columns and col in recommender.category_encoders:
#                 # Transform using the already-fitted encoder
#                 encoded = recommender.category_encoders[col].transform(
#                     unique_products_df[col].fillna('Unknown')
#                 )
#                 unique_product_categorical.append(encoded)
        
#         unique_product_categorical = np.column_stack(unique_product_categorical) if unique_product_categorical else np.zeros((len(unique_products_df), 1))
        
#         # Re-scale numerical features for unique products
#         product_numerical_columns = ['loves_count', 'price_usd_y', 'discount_pct']
#         product_numerical_columns = [col for col in product_numerical_columns if col in unique_products_df.columns]
#         if not product_numerical_columns:
#             unique_products_df['product_num_dummy'] = 0
#             product_numerical_columns = ['product_num_dummy']
        
#         unique_product_numerical = unique_products_df[product_numerical_columns].fillna(0).values
#         unique_product_numerical = recommender.product_numerical_scaler.transform(unique_product_numerical)
        
#         # Generate embeddings for all users and unique products
#         user_embeddings, _ = recommender.generate_embeddings(
#             user_padded, user_categorical, user_numerical,
#             user_padded, user_categorical, user_numerical  # Dummy values, won't be used
#         )
        
#         _, product_embeddings = recommender.generate_embeddings(
#             user_padded[:1], user_categorical[:1], user_numerical[:1],  # Dummy values, won't be used
#             unique_product_padded, unique_product_categorical, unique_product_numerical
#         )
        
#         # Generate example recommendations for a few users
#         num_examples = min(5, len(user_embeddings))
#         for i in range(num_examples):
#             user_idx = i  # Using users as examples
#             user_embedding = user_embeddings[user_idx:user_idx+1]
            
#             logger.info(f"Generating recommendations for user {user_idx}")
            
#             try:
#                 recommendations = recommender.get_recommendations(
#                     user_embedding, 
#                     product_embeddings,
#                     unique_products_df
#                 )
                
#                 print(f"Recommendations for user {user_idx}:")
#                 print(recommendations)
#                 print("\n")
#             except Exception as e:
#                 logger.error(f"Error generating recommendations for user {user_idx}: {str(e)}")
        
#         # Save models
#         recommender.save_models()
        
#         logger.info("Recommendation system completed successfully")
        
#     except Exception as e:
#         logger.error(f"Error in main function: {str(e)}")
#         import traceback
#         logger.error(traceback.format_exc())
#         raise


# if __name__ == "__main__":
#     main()

In [9]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Dense, Embedding, Flatten, Concatenate, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import pickle
import os
import logging
import re

# Set up simple logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('beauty_recommender')

class SimpleBeautyRecommender:
    """Simplified beauty product recommendation system"""
    
    def __init__(self, embedding_dim=64, tower_dim=32, max_words=5000, max_seq_len=100):
        """Initialize the recommender with simplified parameters"""
        self.embedding_dim = embedding_dim
        self.tower_dim = tower_dim
        self.max_words = max_words
        self.max_seq_len = max_seq_len
        
        # Core components
        self.user_tokenizer = None
        self.product_tokenizer = None
        self.encoders = {}
        self.user_scaler = None
        self.product_scaler = None
        self.model = None
        self.user_model = None
        self.product_model = None
        
        # Feature dimensions
        self.feature_dims = {}
        
        # Added: Store product_ids to ensure proper mapping
        self.product_ids = None
    
    def load_data(self, product_path, review_path):
        """Load product and review data from CSV files"""
        logger.info(f"Loading data from {product_path} and {review_path}")
        
        # Added: Error handling for file loading
        try:
            products_df = pd.read_csv(product_path, encoding='ISO-8859-1')
            reviews_df = pd.read_csv(review_path)
        except Exception as e:
            logger.error(f"Error loading data: {e}")
            raise
        
        # Calculate discount percentage if price data available
        if 'price_usd' in products_df.columns and 'sale_price_usd' in products_df.columns:
            products_df['discount_pct'] = products_df.apply(
                lambda row: ((row['price_usd'] - row['sale_price_usd']) / row['price_usd'] * 100) 
                if pd.notna(row['price_usd']) and pd.notna(row['sale_price_usd']) and row['price_usd'] > 0 
                else 0, 
                axis=1
            )
        else:
            products_df['discount_pct'] = 0
            
        # Added: Log data dimensions
        logger.info(f"Loaded {len(products_df)} products and {len(reviews_df)} reviews")
            
        return products_df, reviews_df
    
    def preprocess_data(self, products_df, reviews_df):
        """Preprocess and merge product and review data"""
        logger.info("Preprocessing data...")
        
        # Ensure product_id is string type for proper merging
        products_df['product_id'] = products_df['product_id'].astype(str)
        reviews_df['product_id'] = reviews_df['product_id'].astype(str)
        
        # Added: Reset indices before merging to avoid index misalignment
        products_df = products_df.reset_index(drop=True)
        reviews_df = reviews_df.reset_index(drop=True)
        
        # Merge product and review data
        merged_df = pd.merge(reviews_df, products_df, on='product_id', how='inner')
        
        # Added: Check if merge was successful
        if len(merged_df) == 0:
            logger.warning("Merge resulted in 0 rows. Check product_id compatibility")
            
        # Handle text fields
        merged_df['review_text'] = merged_df['review_text'].fillna('')
        merged_df['review_title'] = merged_df['review_title'].fillna('')
        
        # Simplify highlights handling
        merged_df['highlights_text'] = ''
        if 'highlights' in merged_df.columns:
            merged_df['highlights_text'] = merged_df['highlights'].fillna('').astype(str)
        
        # Fill numerical fields
        numerical_cols = ['rating', 'price_usd', 'discount_pct']
        for col in numerical_cols:
            if col in merged_df.columns:
                merged_df[col] = merged_df[col].fillna(0)
        
        # Handle categorical data
        categorical_cols = ['brand_name', 'primary_category', 'skin_type']
        for col in categorical_cols:
            if col in merged_df.columns:
                merged_df[col] = merged_df[col].fillna('Unknown')
        
        # Clean and combine text
        # User text = review text + review title
        merged_df['user_text'] = self._clean_text(
            merged_df['review_text'] + ' ' + merged_df['review_title']
        )
        
        # Product text = product name + highlights
        product_name_col = 'product_name'
        if product_name_col not in merged_df.columns and 'product_name_x' in merged_df.columns:
            product_name_col = 'product_name_x'
            
        merged_df['product_text'] = self._clean_text(
            merged_df[product_name_col].fillna('') + ' ' + merged_df['highlights_text']
        )
        
        # Added: Reset index after all preprocessing to ensure clean indices
        merged_df = merged_df.reset_index(drop=True)
        
        # Added: Log result
        logger.info(f"Preprocessing complete. Final dataset has {len(merged_df)} rows")
        
        return merged_df
    
    def _clean_text(self, text_series):
        """Basic text cleaning"""
        def clean(text):
            if not isinstance(text, str):
                return ''
            # Convert to lowercase
            text = text.lower()
            # Remove special characters
            text = re.sub(r'[^\w\s]', ' ', text)
            # Remove extra whitespace
            text = re.sub(r'\s+', ' ', text)
            return text.strip()
        
        return text_series.apply(clean)
    
    def prepare_features(self, df):
        """Prepare all features in one method"""
        logger.info("Preparing features...")
        
        # Added: Ensure df has proper index
        df = df.reset_index(drop=True)
        
        # 1. Text features
        # Tokenize user text
        if self.user_tokenizer is None:
            self.user_tokenizer = Tokenizer(num_words=self.max_words)
            self.user_tokenizer.fit_on_texts(df['user_text'])
            
        user_sequences = self.user_tokenizer.texts_to_sequences(df['user_text'])
        user_padded = pad_sequences(user_sequences, maxlen=self.max_seq_len, padding='post')
        
        # Tokenize product text
        if self.product_tokenizer is None:
            self.product_tokenizer = Tokenizer(num_words=self.max_words)
            self.product_tokenizer.fit_on_texts(df['product_text'])
            
        product_sequences = self.product_tokenizer.texts_to_sequences(df['product_text'])
        product_padded = pad_sequences(product_sequences, maxlen=self.max_seq_len, padding='post')
        
        # 2. Categorical features
        # User categorical features
        user_categorical_cols = ['skin_type']
        user_categorical_features = []
        
        for col in user_categorical_cols:
            if col in df.columns:
                if col not in self.encoders:
                    encoder = LabelEncoder()
                    encoder.fit(df[col].fillna('Unknown'))
                    self.encoders[col] = encoder
                    
                # Added: Handle unknown categories by defaulting to 'Unknown'
                df[col] = df[col].fillna('Unknown')
                # Convert categories that weren't seen during training to 'Unknown'
                if hasattr(self.encoders[col], 'classes_'):
                    mask = ~df[col].isin(self.encoders[col].classes_)
                    if mask.any():
                        logger.warning(f"Found {mask.sum()} unknown values in {col}, replacing with 'Unknown'")
                        df.loc[mask, col] = 'Unknown'
                
                encoded = self.encoders[col].transform(df[col])
                user_categorical_features.append(encoded)
        
        # Product categorical features
        product_categorical_cols = ['primary_category', 'brand_name']
        if 'brand_name' not in df.columns and 'brand_name_x' in df.columns:
            product_categorical_cols = ['primary_category', 'brand_name_x']
            
        product_categorical_features = []
        
        for col in product_categorical_cols:
            if col in df.columns:
                if col not in self.encoders:
                    encoder = LabelEncoder()
                    encoder.fit(df[col].fillna('Unknown'))
                    self.encoders[col] = encoder
                
                # Added: Handle unknown categories
                df[col] = df[col].fillna('Unknown')
                if hasattr(self.encoders[col], 'classes_'):
                    mask = ~df[col].isin(self.encoders[col].classes_)
                    if mask.any():
                        logger.warning(f"Found {mask.sum()} unknown values in {col}, replacing with 'Unknown'")
                        df.loc[mask, col] = 'Unknown'
                        
                encoded = self.encoders[col].transform(df[col])
                product_categorical_features.append(encoded)
        
        # Stack features
        user_categorical = np.column_stack(user_categorical_features) if user_categorical_features else np.zeros((len(df), 1))
        product_categorical = np.column_stack(product_categorical_features) if product_categorical_features else np.zeros((len(df), 1))
        
        # 3. Numerical features
        # User numerical features
        user_numerical_cols = ['rating']
        if 'rating' not in df.columns and 'rating_x' in df.columns:
            user_numerical_cols = ['rating_x']
            
        user_numerical = df[user_numerical_cols].fillna(0).values
        
        # Product numerical features
        product_numerical_cols = ['price_usd', 'discount_pct']
        if 'price_usd' not in df.columns and 'price_usd_y' in df.columns:
            product_numerical_cols = ['price_usd_y', 'discount_pct']
            
        product_numerical = df[product_numerical_cols].fillna(0).values
        
        # Scale numerical features
        if self.user_scaler is None:
            self.user_scaler = StandardScaler()
            user_numerical = self.user_scaler.fit_transform(user_numerical)
        else:
            user_numerical = self.user_scaler.transform(user_numerical)
            
        if self.product_scaler is None:
            self.product_scaler = StandardScaler()
            product_numerical = self.product_scaler.fit_transform(product_numerical)
        else:
            product_numerical = self.product_scaler.transform(product_numerical)
        
        # Store feature dimensions
        self.feature_dims['user_text'] = user_padded.shape[1]
        self.feature_dims['user_categorical'] = user_categorical.shape[1]
        self.feature_dims['user_numerical'] = user_numerical.shape[1]
        self.feature_dims['product_text'] = product_padded.shape[1]
        self.feature_dims['product_categorical'] = product_categorical.shape[1]
        self.feature_dims['product_numerical'] = product_numerical.shape[1]
        
        # Added: Store product_ids to map embeddings to products later
        if 'product_id' in df.columns:
            self.product_ids = df['product_id'].values
        
        return {
            'user_text': user_padded,
            'user_categorical': user_categorical,
            'user_numerical': user_numerical,
            'product_text': product_padded,
            'product_categorical': product_categorical,
            'product_numerical': product_numerical
        }
    
    def build_model(self):
        """Build a simplified dual tower model"""
        logger.info("Building dual tower model...")
        
        # Check feature dimensions are set
        if not self.feature_dims:
            raise ValueError("Feature dimensions not set. Run prepare_features first.")
            
        # User tower
        # Text input
        user_text_input = Input(shape=(self.feature_dims['user_text'],), name='user_text_input')
        user_embedding = Embedding(
            input_dim=min(len(self.user_tokenizer.word_index) + 1, self.max_words),
            output_dim=self.embedding_dim
        )(user_text_input)
        user_text_features = Flatten()(user_embedding)
        
        # Categorical input
        user_categorical_input = Input(shape=(self.feature_dims['user_categorical'],), name='user_categorical_input')
        user_categorical_features = Dense(16, activation='relu')(user_categorical_input)
        
        # Numerical input
        user_numerical_input = Input(shape=(self.feature_dims['user_numerical'],), name='user_numerical_input')
        user_numerical_features = Dense(16, activation='relu')(user_numerical_input)
        
        # Combine user features
        user_combined = Concatenate()([
            user_text_features,
            user_categorical_features,
            user_numerical_features
        ])
        
        user_dense = Dense(64, activation='relu')(user_combined)
        user_dropout = Dropout(0.2)(user_dense)
        user_dense2 = Dense(self.tower_dim, activation='relu')(user_dropout)
        user_tower = tf.math.l2_normalize(user_dense2, axis=1)
        
        # Product tower
        # Text input
        product_text_input = Input(shape=(self.feature_dims['product_text'],), name='product_text_input')
        product_embedding = Embedding(
            input_dim=min(len(self.product_tokenizer.word_index) + 1, self.max_words),
            output_dim=self.embedding_dim
        )(product_text_input)
        product_text_features = Flatten()(product_embedding)
        
        # Categorical input
        product_categorical_input = Input(shape=(self.feature_dims['product_categorical'],), name='product_categorical_input')
        product_categorical_features = Dense(16, activation='relu')(product_categorical_input)
        
        # Numerical input
        product_numerical_input = Input(shape=(self.feature_dims['product_numerical'],), name='product_numerical_input')
        product_numerical_features = Dense(16, activation='relu')(product_numerical_input)
        
        # Combine product features
        product_combined = Concatenate()([
            product_text_features,
            product_categorical_features,
            product_numerical_features
        ])
        
        product_dense = Dense(64, activation='relu')(product_combined)
        product_dropout = Dropout(0.2)(product_dense)
        product_dense2 = Dense(self.tower_dim, activation='relu')(product_dropout)
        product_tower = tf.math.l2_normalize(product_dense2, axis=1)
        
        # Similarity calculation
        dot_product = tf.reduce_sum(tf.multiply(user_tower, product_tower), axis=1, keepdims=True)
        
        # Build main model
        self.model = Model(
            inputs=[
                user_text_input,
                user_categorical_input,
                user_numerical_input,
                product_text_input,
                product_categorical_input,
                product_numerical_input
            ],
            outputs=dot_product
        )
        
        # Build embedding models
        self.user_model = Model(
            inputs=[
                user_text_input,
                user_categorical_input,
                user_numerical_input
            ],
            outputs=user_tower
        )
        
        self.product_model = Model(
            inputs=[
                product_text_input,
                product_categorical_input,
                product_numerical_input
            ],
            outputs=product_tower
        )
        
        # Compile model
        self.model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        
        return self.model
    
    def create_training_data(self, features, negative_ratio=3, max_samples=None):
        """Create positive and negative samples for training"""
        logger.info(f"Creating training data with negative_ratio={negative_ratio}")
        
        n_samples = len(features['user_text'])
        if max_samples and max_samples < n_samples:
            n_samples = max_samples
        
        # Total samples including negative samples
        total_samples = n_samples * (1 + negative_ratio)
        
        # Added: Check array dimensions
        for key, feat in features.items():
            logger.info(f"Feature '{key}' shape: {feat.shape}")
        
        # Allocate arrays
        X_user_text = np.zeros((total_samples, features['user_text'].shape[1]), dtype=np.int32)
        X_user_cat = np.zeros((total_samples, features['user_categorical'].shape[1]), dtype=np.float32)
        X_user_num = np.zeros((total_samples, features['user_numerical'].shape[1]), dtype=np.float32)
        X_prod_text = np.zeros((total_samples, features['product_text'].shape[1]), dtype=np.int32)
        X_prod_cat = np.zeros((total_samples, features['product_categorical'].shape[1]), dtype=np.float32)
        X_prod_num = np.zeros((total_samples, features['product_numerical'].shape[1]), dtype=np.float32)
        y = np.zeros((total_samples, 1), dtype=np.float32)
        
        sample_idx = 0
        
        # Add positive and negative samples
        for i in range(n_samples):
            # Positive example (matching user and product)
            X_user_text[sample_idx] = features['user_text'][i]
            X_user_cat[sample_idx] = features['user_categorical'][i]
            X_user_num[sample_idx] = features['user_numerical'][i]
            X_prod_text[sample_idx] = features['product_text'][i]
            X_prod_cat[sample_idx] = features['product_categorical'][i]
            X_prod_num[sample_idx] = features['product_numerical'][i]
            y[sample_idx] = 1  # Positive label
            sample_idx += 1
            
            # Negative examples (user with random products)
            for _ in range(negative_ratio):
                # Select a random product
                neg_idx = np.random.randint(0, n_samples)
                while neg_idx == i and n_samples > 1:
                    neg_idx = np.random.randint(0, n_samples)
                
                # Same user, different product
                X_user_text[sample_idx] = features['user_text'][i]
                X_user_cat[sample_idx] = features['user_categorical'][i]
                X_user_num[sample_idx] = features['user_numerical'][i]
                X_prod_text[sample_idx] = features['product_text'][neg_idx]
                X_prod_cat[sample_idx] = features['product_categorical'][neg_idx]
                X_prod_num[sample_idx] = features['product_numerical'][neg_idx]
                y[sample_idx] = 0  # Negative label
                sample_idx += 1
        
        # Create training data array
        X = [
            X_user_text,
            X_user_cat,
            X_user_num,
            X_prod_text,
            X_prod_cat,
            X_prod_num
        ]
        
        # Added: Log shapes
        logger.info(f"Created training data with {len(y)} total samples")
        
        return X, y
    
    def train(self, X, y, validation_split=0.2, epochs=5, batch_size=64):
        """Train the recommendation model"""
        logger.info(f"Training model with {len(y)} samples")
        
        # Split training and validation
        indices = np.arange(len(y))
        train_idx, val_idx = train_test_split(
            indices, 
            test_size=validation_split,
            random_state=42,
            stratify=y
        )
        
        # Use indices to split data
        X_train = [x[train_idx] for x in X]
        X_val = [x[val_idx] for x in X]
        y_train = y[train_idx]
        y_val = y[val_idx]
        
        # Define early stopping
        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=2,
            restore_best_weights=True
        )
        
        # Train model
        history = self.model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[early_stopping]
        )
        
        return history
    
    def generate_embeddings(self, features, batch_size=64):
        """Generate embeddings for users and products"""
        logger.info("Generating embeddings...")
        
        # Check model is built
        if self.user_model is None or self.product_model is None:
            raise ValueError("Models not built. Run build_model first.")
        
        # Added: Log input shapes
        for key, feat in features.items():
            logger.info(f"Feature '{key}' shape: {feat.shape}")
            
        # User embeddings
        user_embeddings = self.user_model.predict(
            [
                features['user_text'],
                features['user_categorical'],
                features['user_numerical']
            ],
            batch_size=batch_size
        )
        
        # Product embeddings
        product_embeddings = self.product_model.predict(
            [
                features['product_text'],
                features['product_categorical'],
                features['product_numerical']
            ],
            batch_size=batch_size
        )
        
        # Added: Log output shapes
        logger.info(f"Generated user embeddings shape: {user_embeddings.shape}")
        logger.info(f"Generated product embeddings shape: {product_embeddings.shape}")
        
        return user_embeddings, product_embeddings
    
    def get_recommendations(self, user_embedding, product_embeddings, df, top_n=5):
        """Get product recommendations for a user"""
        logger.info(f"Getting top {top_n} recommendations")
        
        # Added: Validate inputs
        if len(product_embeddings) != len(df):
            logger.warning(f"Mismatch between product embeddings ({len(product_embeddings)}) and DataFrame ({len(df)})")
            # Use the minimum length to avoid index errors
            min_len = min(len(product_embeddings), len(df))
            product_embeddings = product_embeddings[:min_len]
            df = df.iloc[:min_len].reset_index(drop=True)
        
        # Calculate similarity scores
        similarity = np.dot(user_embedding, product_embeddings.T)[0]
        
        # Added: Ensure top_n doesn't exceed data size
        top_n = min(top_n, len(df))
        
        # Get indices of top N products
        top_indices = np.argsort(similarity)[::-1][:top_n]
        
        # Added: Safety check for indices
        top_indices = [idx for idx in top_indices if 0 <= idx < len(df)]
        
        # Select recommended products
        if len(top_indices) > 0:
            recommendations = df.iloc[top_indices].copy()
            # Add similarity scores to recommendations
            recommendations['similarity_score'] = similarity[top_indices]
            recommendations = recommendations.sort_values('similarity_score', ascending=False)
        else:
            logger.warning("No valid recommendations found")
            recommendations = pd.DataFrame()
        
        return recommendations
    
    def save(self, path='.'):
        """Save models and components"""
        logger.info(f"Saving models to {path}")
        
        os.makedirs(path, exist_ok=True)
        
        try:
            # Save models
            self.model.save(os.path.join(path, 'model.h5'))
            self.user_model.save(os.path.join(path, 'user_model.h5'))
            self.product_model.save(os.path.join(path, 'product_model.h5'))
            
            # Save tokenizers and other components
            with open(os.path.join(path, 'components.pkl'), 'wb') as f:
                pickle.dump({
                    'user_tokenizer': self.user_tokenizer,
                    'product_tokenizer': self.product_tokenizer,
                    'encoders': self.encoders,
                    'user_scaler': self.user_scaler,
                    'product_scaler': self.product_scaler,
                    'feature_dims': self.feature_dims,
                    'product_ids': self.product_ids  # Added: Save product IDs
                }, f)
            
            logger.info("Models and components saved successfully")
        except Exception as e:
            logger.error(f"Error saving models: {e}")
            raise
    
    def load(self, path='.'):
        """Load models and components"""
        logger.info(f"Loading models from {path}")
        
        try:
            # Load models
            self.model = tf.keras.models.load_model(os.path.join(path, 'model.h5'))
            self.user_model = tf.keras.models.load_model(os.path.join(path, 'user_model.h5'))
            self.product_model = tf.keras.models.load_model(os.path.join(path, 'product_model.h5'))
            
            # Load components
            with open(os.path.join(path, 'components.pkl'), 'rb') as f:
                components = pickle.load(f)
                self.user_tokenizer = components['user_tokenizer']
                self.product_tokenizer = components['product_tokenizer']
                self.encoders = components['encoders']
                self.user_scaler = components['user_scaler']
                self.product_scaler = components['product_scaler']
                self.feature_dims = components['feature_dims']
                # Added: Load product IDs if available
                if 'product_ids' in components:
                    self.product_ids = components['product_ids']
                    
            logger.info("Models and components loaded successfully")
        except Exception as e:
            logger.error(f"Error loading models: {e}")
            raise


def main():
    """Simple example of how to use the system"""
    # Initialize recommender
    recommender = SimpleBeautyRecommender()
    
    try:
        # Load and preprocess data
        products_df, reviews_df = recommender.load_data(
            '../data/product_info.csv', 
            '../data/reviews_0-250_chunk_0.csv'
        )
        
        # Added: Log data dimensions
        logger.info(f"Loaded product data: {products_df.shape}, review data: {reviews_df.shape}")
        
        # Preprocess data
        merged_df = recommender.preprocess_data(products_df, reviews_df)
        
        # Create unique product dataset for recommendations
        # Added: Move this before prepare_features for clearer logic
        unique_products = merged_df.drop_duplicates(subset=['product_id']).reset_index(drop=True)
        logger.info(f"Unique products: {len(unique_products)}")
        
        # Prepare features for training
        features = recommender.prepare_features(merged_df)
        
        # Build model
        recommender.build_model()
        
        # Create training data
        X, y = recommender.create_training_data(features, max_samples=5000)
        
        # Train model
        recommender.train(X, y, epochs=5)
        
        # Prepare features for unique products - this ensures alignment
        unique_features = recommender.prepare_features(unique_products)
        
        # Generate embeddings specifically for unique products
        _, product_embeddings = recommender.generate_embeddings(unique_features)
        
        # Generate user embeddings from the original features
        user_embeddings, _ = recommender.generate_embeddings(features)
        
        # Verify dimensions
        logger.info(f"user_embeddings shape: {user_embeddings.shape}")
        logger.info(f"product_embeddings shape: {product_embeddings.shape}")
        logger.info(f"unique_products length: {len(unique_products)}")
        
        # Added: Safety check to ensure indices match
        if len(product_embeddings) != len(unique_products):
            logger.warning(f"Mismatch between embeddings and products. Trimming to match.")
            min_len = min(len(product_embeddings), len(unique_products))
            product_embeddings = product_embeddings[:min_len]
            unique_products = unique_products.iloc[:min_len].reset_index(drop=True)
            
        # Get recommendations for a sample user
        user_idx = 0
        if user_idx < len(user_embeddings):
            recommendations = recommender.get_recommendations(
                user_embeddings[user_idx:user_idx+1],
                product_embeddings,
                unique_products
            )
            
            if not recommendations.empty:
                print("Recommendations:")
                display_cols = ['product_id', 'similarity_score']
                product_name_col = 'product_name'
                if product_name_col not in recommendations.columns:
                    product_name_col = next((col for col in recommendations.columns if 'product_name' in col), None)
                
                brand_col = 'brand_name'
                if brand_col not in recommendations.columns:
                    brand_col = next((col for col in recommendations.columns if 'brand_name' in col), None)
                
                price_col = 'price_usd'
                if price_col not in recommendations.columns:
                    price_col = next((col for col in recommendations.columns if 'price_usd' in col), None)
                
                if product_name_col:
                    display_cols.insert(1, product_name_col)
                if brand_col:
                    display_cols.append(brand_col)
                if price_col:
                    display_cols.append(price_col)
                
                print(recommendations[display_cols])
            else:
                print("No recommendations found.")
        else:
            print(f"User index {user_idx} is out of bounds. Max index is {len(user_embeddings)-1}")
        
        # Save the model
        recommender.save('models')
        print("Model saved successfully.")
        
    except Exception as e:
        logger.error(f"An error occurred: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()

2025-03-06 14:50:31,774 - beauty_recommender - INFO - Loading data from ../data/product_info.csv and ../data/reviews_0-250_chunk_0.csv
C:\Users\hp\AppData\Local\Temp\ipykernel_33288\1416662772.py:53: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews_df = pd.read_csv(review_path)
2025-03-06 14:50:32,835 - beauty_recommender - INFO - Loaded 8494 products and 100355 reviews
2025-03-06 14:50:32,837 - beauty_recommender - INFO - Loaded product data: (8494, 28), review data: (100355, 19)
2025-03-06 14:50:32,837 - beauty_recommender - INFO - Preprocessing data...
2025-03-06 14:50:36,972 - beauty_recommender - INFO - Preprocessing complete. Final dataset has 100355 rows
2025-03-06 14:50:37,030 - beauty_recommender - INFO - Unique products: 15
2025-03-06 14:50:37,031 - beauty_recommender - INFO - Preparing features...
2025-03-06 14:50:47,412 - beauty_recommender - INFO - Building dual tower model...
2025-03-06 14:50:47,890 - beauty_rec

Epoch 1/5
250/250 [==============================] - 6s 16ms/step - loss: 0.5731 - accuracy: 0.7491 - val_loss: 0.5700 - val_accuracy: 0.7500
Epoch 2/5
250/250 [==============================] - 4s 15ms/step - loss: 0.5648 - accuracy: 0.7500 - val_loss: 0.5654 - val_accuracy: 0.7500
Epoch 3/5
250/250 [==============================] - 3s 14ms/step - loss: 0.5638 - accuracy: 0.7500 - val_loss: 0.5656 - val_accuracy: 0.7500
Epoch 4/5
250/250 [==============================] - 3s 14ms/step - loss: 0.5634 - accuracy: 0.7500 - val_loss: 0.5681 - val_accuracy: 0.7500


2025-03-06 14:51:04,749 - beauty_recommender - INFO - Preparing features...
2025-03-06 14:51:04,772 - beauty_recommender - INFO - Generating embeddings...
2025-03-06 14:51:04,773 - beauty_recommender - INFO - Feature 'user_text' shape: (15, 100)
2025-03-06 14:51:04,774 - beauty_recommender - INFO - Feature 'user_categorical' shape: (15, 1)
2025-03-06 14:51:04,775 - beauty_recommender - INFO - Feature 'user_numerical' shape: (15, 1)
2025-03-06 14:51:04,777 - beauty_recommender - INFO - Feature 'product_text' shape: (15, 100)
2025-03-06 14:51:04,778 - beauty_recommender - INFO - Feature 'product_categorical' shape: (15, 2)
2025-03-06 14:51:04,779 - beauty_recommender - INFO - Feature 'product_numerical' shape: (15, 2)


1/1 [==============================] - 0s 124ms/step


2025-03-06 14:51:05,093 - beauty_recommender - INFO - Generated user embeddings shape: (15, 32)
2025-03-06 14:51:05,094 - beauty_recommender - INFO - Generated product embeddings shape: (15, 32)
2025-03-06 14:51:05,094 - beauty_recommender - INFO - Generating embeddings...
2025-03-06 14:51:05,096 - beauty_recommender - INFO - Feature 'user_text' shape: (100355, 100)
2025-03-06 14:51:05,096 - beauty_recommender - INFO - Feature 'user_categorical' shape: (100355, 1)
2025-03-06 14:51:05,097 - beauty_recommender - INFO - Feature 'user_numerical' shape: (100355, 1)
2025-03-06 14:51:05,097 - beauty_recommender - INFO - Feature 'product_text' shape: (100355, 100)
2025-03-06 14:51:05,098 - beauty_recommender - INFO - Feature 'product_categorical' shape: (100355, 2)
2025-03-06 14:51:05,098 - beauty_recommender - INFO - Feature 'product_numerical' shape: (100355, 2)


...

2025-03-06 14:51:15,786 - beauty_recommender - INFO - Generated user embeddings shape: (100355, 32)
2025-03-06 14:51:15,787 - beauty_recommender - INFO - Generated product embeddings shape: (100355, 32)
2025-03-06 14:51:15,788 - beauty_recommender - INFO - user_embeddings shape: (100355, 32)
2025-03-06 14:51:15,789 - beauty_recommender - INFO - product_embeddings shape: (15, 32)
2025-03-06 14:51:15,789 - beauty_recommender - INFO - unique_products length: 15
2025-03-06 14:51:15,790 - beauty_recommender - INFO - Getting top 5 recommendations
2025-03-06 14:51:15,808 - beauty_recommender - INFO - Saving models to models
d:\Program\Anaconda\envs\tensorflow\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Recommendations:
   product_id                                     product_name_x  \
1     P420652  Lip Sleeping Mask Intense Hydration with Vitam...   
4     P248407               Ultra Repair Cream Intense Hydration   
11    P441644                Mini Superfood Antioxidant Cleanser   
8     P450271  Green Clean Makeup Meltaway Cleansing Balm Lim...   
10    P411387                     Superfood Antioxidant Cleanser   

    similarity_score         brand_name_x  price_usd_x  
1           0.221956              LANEIGE         24.0  
4           0.221939     First Aid Beauty         38.0  
11          0.221255  Youth To The People         14.0  
8           0.220518              Farmacy         60.0  
10          0.220467  Youth To The People         39.0  


2025-03-06 14:51:16,060 - tensorflow - WARNING - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


2025-03-06 14:51:16,080 - tensorflow - WARNING - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.
2025-03-06 14:51:16,136 - beauty_recommender - INFO - Models and components saved successfully


Model saved successfully.
